In [1]:
import json
import os
import numpy as np
import pandas as pd

data_dir = "../data/raw/"
all_data = []

for file in os.listdir(data_dir):
    if file.endswith(".json"):
        with open(os.path.join(data_dir, file)) as f:
            all_data.append(json.load(f))

print(f"Loaded {len(all_data)} users")
for p in all_data:
    print(f"  → {p['username']}: {len(p['attempts'])} attempts")

Loaded 5 users
  → Diya: 15 attempts
  → Jash: 15 attempts
  → Khadir: 15 attempts
  → RITI: 15 attempts
  → rugvedd: 15 attempts


In [2]:
def build_feature_vector(attempt, all_attempts_for_user):
    f = attempt['features']
    
    row = {
        'mean_dwell':      f['mean_dwell'],
        'std_dwell':       f['std_dwell'],
        'mean_flight':     f['mean_flight'],
        'std_flight':      f['std_flight'],
        'total_time_ms':   f['total_time_ms'],
        'total_keys':      f['total_keys'],
        'backspace_count': f['backspace_count'],
        'backspace_rate':  f['backspace_count'] / max(f['total_keys'], 1),
        'typing_speed':    f['total_keys'] / max(f['total_time_ms'], 1) * 1000,
    }
    
    target_keys = list(set('thequickbrownfoxjumpsoverthelazydog'))
    for key in target_keys:
        val = f['dwell_per_key'].get(key, np.nan)
        row[f'dwell_{key}'] = val
    
    common_digraphs = ['th', 'he', 'er', 'qu', 'ow', 'br', 'fo', 'ox', 
                       'ju', 'um', 'ov', 'la', 'az', 'zy', 'do', 'og']
    
    digraph_lookup = {}
    for ft in f['flight_times']:
        if ft['from'] and ft['to']:
            pair = ft['from'] + ft['to']
            if pair not in digraph_lookup:
                digraph_lookup[pair] = []
            digraph_lookup[pair].append(ft['time'])
    
    for digraph in common_digraphs:
        times = digraph_lookup.get(digraph, [np.nan])
        row[f'digraph_{digraph}'] = np.nanmean(times)
    
    return row

rows = []
for person in all_data:
    for attempt in person['attempts']:
        row = build_feature_vector(attempt, person['attempts'])
        row['user'] = person['username']
        rows.append(row)

df = pd.DataFrame(rows)
print(f"Feature matrix shape: {df.shape}")
print(f"\nUsers: {df['user'].value_counts().to_dict()}")
df.head()

Feature matrix shape: (75, 52)

Users: {'Diya': 15, 'Jash': 15, 'Khadir': 15, 'RITI': 15, 'rugvedd': 15}


C:\Users\jashv\AppData\Local\Temp\ipykernel_30672\1526331612.py:34: RuntimeWarning: Mean of empty slice
  row[f'digraph_{digraph}'] = np.nanmean(times)


,mean_dwell,std_dwell,mean_flight,std_flight,total_time_ms,total_keys,backspace_count,backspace_rate,typing_speed,dwell_l,...,digraph_ox,digraph_ju,digraph_um,digraph_ov,digraph_la,digraph_az,digraph_zy,digraph_do,digraph_og,user
0,79.153623,27.374364,242.550000,217.657107,20386.4,69,3,0.043478,3.384609,92.7,...,321.5,150.7,NaN,217.8,64.8,206.4,43.7,191.8,195.9,Diya
1,90.242857,30.605476,194.911290,199.912470,15621.3,63,0,0.000000,4.032955,132.1,...,234.1,55.3,108.8,199.7,51.6,277.0,57.0,39.4,104.5,Diya
2,82.667692,29.537705,181.635938,312.682286,15659.1,65,1,0.015385,4.150941,88.3,...,166.5,102.0,128.6,173.9,60.9,323.1,228.2,24.3,106.6,Diya
3,85.388889,30.563107,142.712903,150.783263,13339.5,63,0,0.000000,4.722816,95.0,...,312.7,112.9,126.4,158.1,32.3,335.2,40.4,14.2,115.6,Diya
4,85.953731,27.994335,153.162121,137.689481,13736.6,67,2,0.029851,4.877481,135.0,...,231.0,152.8,158.1,146.7,30.4,310.1,NaN,NaN,82.8,Diya


In [3]:
nan_counts = df.isna().sum()
print("Columns with NaN values:")
print(nan_counts[nan_counts > 0])

df_clean = df.copy()
feature_cols = [c for c in df.columns if c != 'user']
df_clean[feature_cols] = df_clean[feature_cols].fillna(
    df_clean[feature_cols].median()
)

print(f"\nAfter cleaning — any NaNs left: {df_clean.isna().sum().sum()}")

Columns with NaN values:
dwell_q        1
digraph_qu    12
digraph_br     5
digraph_fo     6
digraph_ox     3
digraph_um     1
digraph_ov     1
digraph_la     6
digraph_az     3
digraph_zy     5
digraph_do     5
digraph_og     6
dtype: int64

After cleaning — any NaNs left: 0


In [4]:
df_clean.to_csv("../data/features.csv", index=False)
print("✅ Feature matrix saved to data/features.csv")
print(f"Shape: {df_clean.shape}")
print(f"\nFeature columns ({len(feature_cols)}):")
print(feature_cols)

✅ Feature matrix saved to data/features.csv
Shape: (75, 52)

Feature columns (51):
['mean_dwell', 'std_dwell', 'mean_flight', 'std_flight', 'total_time_ms', 'total_keys', 'backspace_count', 'backspace_rate', 'typing_speed', 'dwell_l', 'dwell_v', 'dwell_k', 'dwell_x', 'dwell_m', 'dwell_f', 'dwell_b', 'dwell_o', 'dwell_j', 'dwell_u', 'dwell_w', 'dwell_h', 'dwell_y', 'dwell_s', 'dwell_i', 'dwell_p', 'dwell_r', 'dwell_g', 'dwell_d', 'dwell_a', 'dwell_q', 'dwell_n', 'dwell_c', 'dwell_t', 'dwell_z', 'dwell_e', 'digraph_th', 'digraph_he', 'digraph_er', 'digraph_qu', 'digraph_ow', 'digraph_br', 'digraph_fo', 'digraph_ox', 'digraph_ju', 'digraph_um', 'digraph_ov', 'digraph_la', 'digraph_az', 'digraph_zy', 'digraph_do', 'digraph_og']
